# This code is a demonstration on model usage
The model used was packaged using 'pickle'

In [11]:
# Importing Libraries
import pandas as pd
import pickle
import numpy as np
import sys
import os
import xgboost as xgb

In [18]:
model_path = 'xgboost_model.pkl' # Defining model path
schema_path = 'input_schema.txt' # Defining schema path
threshold = 0.3 # Optimal threshold found using RandomizedSearchCV in the source code

def demo():
    # Loading model
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    model_features = model.get_booster().feature_names
    print(f"Model expects {len(model_features)} features")

    # Loading schema
    parsed_features = []
    try:
        with open(schema_path, 'r') as f:
            lines = f.readlines()

        start_idx = next(i for i, l in enumerate(lines) if l.strip() == "Column Details:") + 1

        for line in lines[start_idx:]:
            if '|' in line:
                parts = [p.strip() for p in line.split('|')]
                if len(parts) >= 2:
                    parsed_features.append(parts[1])

        print(f"Parsed {len(parsed_features)} features from schema")

        # Comparing schema vs model
        extra = set(parsed_features) - set(model_features)
        missing = set(model_features) - set(parsed_features)

        if extra:
          print(f"Extra in TXT (ignored): {extra}")
        if missing:
          print(f"Missing from TXT: {missing}")

    except Exception as e:
        print(f"[WARNING] Schema load failed: {e}")

    # Generating realistic test data 
    n = len(model_features)
    rows = []

    # Sparse rows
    for _ in range(5):
        row = np.zeros(n)
        idx = np.random.choice(n, max(1, int(0.1 * n)), replace=False)
        row[idx] = np.random.uniform(0.1, 1.0, size=len(idx))
        rows.append(row)

    # Noisy rows
    for _ in range(3):
        rows.append(np.random.beta(2, 5, size=n))

    # Edge cases
    rows.append(np.zeros(n))
    rows.append(np.ones(n))

    demo_df = pd.DataFrame(rows, columns=model_features)
    demo_df = demo_df.reindex(columns=model_features, fill_value=np.nan)
    print("Not 0 counts per row:")
    print(((demo_df != 0) & (~demo_df.isna())).sum(axis=1))

    print(f"\nModel running on demo ({len(demo_df)} rows)")

    # Predictions
    probs = model.predict_proba(demo_df)[:, 1]
    preds = (probs >= threshold).astype(int)

    results = pd.DataFrame({
        "risk_score": probs,
        "prediction": preds
    })

    print("\nResults")
    print(results)


if __name__ == "__main__":
    demo()


Model expects 169 features
Parsed 169 features from schema
Not 0 counts per row:
0     16
1     16
2     16
3     16
4     16
5    169
6    169
7    169
8      0
9    169
dtype: int64

Model running on demo (10 rows)

Results
   risk_score  prediction
0    0.999707           1
1    0.999707           1
2    0.999707           1
3    0.999707           1
4    0.999707           1
5    0.999707           1
6    0.999707           1
7    0.999707           1
8    0.999707           1
9    0.999707           1
